<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Renew a Slice Reservation

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**What this notebook covers:** This notebook demonstrates how to extend (renew) the lease on an existing FABRIC slice. Every slice has an expiration date -- when it arrives, the slice is automatically deleted. Renewing lets you extend your experiment without rebuilding from scratch.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Understand how FABRIC **slice leases** and expiration dates work
2. Calculate a new **end date** using Python's `datetime` module
3. **Renew** a slice's lease with `slice.renew(end_date)`
4. **Verify** the new lease end date after renewal

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you should:

1. Have completed the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Have an **existing active slice** to renew (create one with [Hello, FABRIC](../hello_fabric/hello_fabric.ipynb) or [Create Slice](../create_slice/create_slice.ipynb))

**Tip:** Replace `'MySlice'` in the examples below with the name of your actual slice.

</div>

## Background: Slice Leases and Expiration

Every FABRIC slice has a **lease** with a start date and an **end date** (expiration). When the lease expires, FABRIC automatically deletes the slice and releases all its resources.


Key points about slice renewal:

- The new end date must be specified in **UTC** timezone
- The date format is `"YYYY-MM-DD HH:MM:SS +0000"`
- There is a **maximum lease duration** set by your project's allocation policy (typically 7-14 days from the current time)
- You can renew a slice **multiple times** as long as the new end date is within the allowed range

<div class="fab-danger">

**Important:** If you set an end date beyond your project's maximum allowed duration, the renewal will fail with an authorization error. Check your project's policies if you encounter this issue.

</div>

---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
# Import the FABlib library
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration
fablib.show_config();

---

## Step 2: Set the Slice Name

Specify which slice you want to renew. Replace `'MySlice'` with the name of your actual slice.

In [ ]:
# Set the name of the slice to renew
slice_name='MySlice'

---

## Step 3: Renew the Slice

Calculate a new end date and call `slice.renew()`. In this example, we extend the lease to **6 days from now**. The end date must be a string in UTC format.

<div class="fab-warning">

**Tip:** Adjust the `timedelta(days=6)` value to set a different lease duration. Common values are 1-14 days, depending on your project's maximum allowed duration.

</div>

In [ ]:
from datetime import datetime
from datetime import timezone
from datetime import timedelta

# Calculate the new end date: current UTC time + 6 days
# Format as a string: "YYYY-MM-DD HH:MM:SS +0000"
end_date = (datetime.now(timezone.utc) + timedelta(days=6)).strftime("%Y-%m-%d %H:%M:%S %z")

try:
    # Retrieve the slice by name
    slice = fablib.get_slice(name=slice_name)

    # Renew the slice with the new end date
    # This sends the renewal request to FABRIC
    slice.renew(end_date)
except Exception as e:
    print(f"Exception: {e}")

---

## Step 4: Verify the New Lease End Date

After renewal, retrieve the slice again and check its `lease_end` property to confirm the new expiration date was applied.

In [ ]:
try:
    # Retrieve the slice again to get the updated lease information
    slice = fablib.get_slice(name=slice_name)
    
    # Print the new lease end date (in UTC)
    print(f"Lease End (UTC)        : {slice.get_lease_end()}")
       
except Exception as e:
    print(f"Exception: {e}")

<div class="fab-success">

**Success!** If the lease end date above matches your requested extension, the renewal was successful. Your slice will now remain active until the new expiration date.

</div>

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `Slice not found` | Slice name is misspelled or was deleted | Run `fablib.list_slices()` to verify available slice names |
| `PDP Authorization check failed` | New end date exceeds project's max allowed duration | Request a shorter extension (e.g., `timedelta(days=3)`) |
| `Renewal failed` | Slice is in an error state | Check slice state with `slice.show()` -- only `StableOK` slices can be renewed |
| End date is in the past | `timedelta` value is negative or zero | Ensure you are adding a positive number of days |
| Lease end date did not change | Renewal request was silently rejected | Check for errors in the renewal response; verify project policies |
| `Token expired` | Authentication token needs refresh | Re-run the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.get_slice(name)` | Retrieve a slice object by name | [get_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_slice) |
| `slice.renew(end_date)` | Extend the slice's lease to a new end date | [renew](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.renew) |
| `slice.get_lease_end()` | Get the current lease end date | [get_lease_end](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.get_lease_end) |

## What's Next?

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **Showing Slice Details** | [show_slice](../slices/show_slice.ipynb) | Inspect all slice properties including lease dates |
| **Listing Slices** | [list_slices](../slices/list_slices.ipynb) | View and filter all your slices |
| **Deleting Slices** | [delete_slice](../delete_slice/delete_slice.ipynb) | Clean up slices you no longer need |
| **Creating Slices** | [create_slice](../create_slice/create_slice.ipynb) | Create new slices with various options |
| **Listing Nodes & Networks** | [list_node_and_networks](../slices/list_node_and_networks.ipynb) | Inspect nodes, networks, interfaces, and components |